In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import yaml

plt.rcParams["figure.dpi"] = 300

def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")


def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


CFG_FILE = find_upwards("config.yaml")
assert CFG_FILE, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(CFG_FILE.read_text())


METHOD = CFG["vpr"]["method"]
RETRIEVAL_METHOD = CFG["retrieval"]["method"]
THRESHOLD = CFG["retrieval"]["threshold"]

PROJECT_ROOT = find_project_root()
RESULT_DIR = PROJECT_ROOT / "results" 
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings"
RETRIEVAL_DIR = RESULT_DIR / "retrieval"
embedding_path = EMBEDDING_DIR / f"{METHOD}_embeddings.npy"
metadata_path = EMBEDDING_DIR / f"{METHOD}_metadata.parquet"

RETRIEVAL_DIR.mkdir(parents = True, exist_ok = True)

embedding_metadata = pd.read_parquet(metadata_path)
database_mask = (embedding_metadata["split"] == "database").to_numpy()
query_mask = (embedding_metadata["split"] == "query").to_numpy()
database_metadata = embedding_metadata[database_mask].reset_index(drop=True)
query_metadata = embedding_metadata[query_mask].reset_index(drop=True)

embeddings = np.load(embedding_path)

database_embeddings = embeddings[database_mask]
query_embeddings = embeddings[query_mask]


retrieval = np.load(RETRIEVAL_DIR / f"{METHOD}_retrieval.npz")
retrieved_indices = retrieval["indices"]
similarities = retrieval["similarities"]


In [ ]:
# harvisine distance
# https://en.wikipedia.org/wiki/Haversine_formula

def haversine_distance(
        lat1,
        lon1,
        lat2,
        lon2
):
    earth_radius = 6_371_000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    diff_lat = lat2 - lat1
    diff_lon = lon2 - lon1

    a = (np.sin(diff_lat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(diff_lon / 2) ** 2)

    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius * c



In [ ]:
def ground_truth(query_index, query_metadata, database_metadata, threshold = THRESHOLD):
    query = query_metadata.iloc[query_index]

    haversine_distances = haversine_distance(
        query["lat"],
        query["lon"],
        database_metadata["lat"].to_numpy(),
        database_metadata["lon"].to_numpy()
    )
    ground = np.where(haversine_distances <= threshold)[0]

    return ground, haversine_distances

In [ ]:
def recall_k( retrieved_indices, ground):

    return int(np.isin(retrieved_indices, ground).any())

In [ ]:
K_VALUES = [1,5,10,20]

recall_results = { k: [] for k in K_VALUES }

for query_index in range(len(query_embeddings)):
    ground, haversine_distances = ground_truth(query_index, query_metadata, database_metadata, threshold= THRESHOLD)

    if len(ground) == 0:
        continue

    query_retrieved_indices = retrieval[query_index]


    for k in K_VALUES:
        recall = recall_k(query_retrieved_indices[:k], ground)
        recall_results[k].append(recall)


for k in K_VALUES:
    recall = np.mean(recall_results[k])
    print(f"Recall@{k}: {recall:.2f}")


ground, distances = ground_truth(0, query_metadata, database_metadata, threshold=float(THRESHOLD))

print("Min distance:", distances.min())
print("Max distance:", distances.max())
print("Ground truth:", ground)

# Detailed Representation for top Kandidates
  
Done by Chatgpt

In [ ]:
import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np


# ============================================================
# 1. Stadtgrenze laden
# ============================================================

CITY_NAME = "Osnabrück, Germany"

city_gdf = ox.geocode_to_gdf(CITY_NAME)

city_gdf = city_gdf[city_gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

if city_gdf.empty:
    raise RuntimeError(f"Keine Stadtgrenze für {CITY_NAME} gefunden.")

city_polygon = city_gdf.geometry.iloc[0]


# ============================================================
# 2. OSM-Straßennetz für ganz Osnabrück laden
# ============================================================

street_net = ox.graph_from_polygon(city_polygon, network_type="drive")


# ============================================================
# 3. Query auswählen
# ============================================================

query_index = 0

query = query_metadata.iloc[query_index]

query_lat = query["lat"]
query_lon = query["lon"]


# Top-K Retrieval-Indizes
K = 10

query_retrieved_indices = retrieval["indices"][query_index]
top_k_indices = query_retrieved_indices[:K]

top_k = database_metadata.iloc[top_k_indices].copy()


# ============================================================
# 4. Abstände zur Query berechnen
# ============================================================

top_k["distance_m"] = haversine_distance(
    query_lat, query_lon, top_k["lat"].to_numpy(), top_k["lon"].to_numpy()
)

print(f"Query: {query['image_id']}")
print()
print(top_k[["image_id", "lat", "lon", "distance_m"]].to_string(index=False))


# ============================================================
# 5. Hilfsfunktion zum Zeichnen
# ============================================================


def plot_retrieval_map(street_net, city_polygon, query, top_k, zoom=False, title=None):
    fig, ax = plt.subplots(figsize=(12, 12))

    # OSM-Straßennetz
    ox.plot_graph(
        street_net,
        ax=ax,
        node_size=0,
        edge_linewidth=0.35,
        edge_color="#cccccc",
        bgcolor="white",
        show=False,
        close=False,
    )

    # --------------------------------------------------------
    # Alle Top-K Retrievals
    # --------------------------------------------------------

    ax.scatter(
        top_k["lon"],
        top_k["lat"],
        s=55,
        c="#1f77b4",
        alpha=0.85,
        edgecolors="white",
        linewidths=0.8,
        zorder=5,
        label=f"Top-{len(top_k)} Retrieval",
    )

    # --------------------------------------------------------
    # Top-1 besonders markieren
    # --------------------------------------------------------

    top1 = top_k.iloc[0]

    ax.scatter(
        top1["lon"],
        top1["lat"],
        s=180,
        facecolors="none",
        edgecolors="red",
        linewidths=2.5,
        zorder=7,
        label="Top-1",
    )

    # --------------------------------------------------------
    # Query markieren
    # --------------------------------------------------------

    ax.scatter(
        query["lon"],
        query["lat"],
        s=220,
        marker="*",
        c="black",
        edgecolors="white",
        linewidths=1.2,
        zorder=8,
        label="Query",
    )

    # --------------------------------------------------------
    # Verbindung Query -> Top-1
    # --------------------------------------------------------

    ax.plot(
        [query["lon"], top1["lon"]],
        [query["lat"], top1["lat"]],
        linestyle="--",
        linewidth=1.2,
        color="red",
        alpha=0.7,
        zorder=6,
    )

    # --------------------------------------------------------
    # Nummern für Ranking
    # --------------------------------------------------------

    for rank, (_, row) in enumerate(top_k.iterrows(), start=1):
        ax.annotate(
            str(rank),
            (row["lon"], row["lat"]),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=9,
            fontweight="bold",
            zorder=9,
        )

    # --------------------------------------------------------
    # Stadtgrenze
    # --------------------------------------------------------

    gpd.GeoSeries([city_polygon], crs="EPSG:4326").boundary.plot(
        ax=ax, color="black", linewidth=1.2, linestyle="--", zorder=4
    )

    # ========================================================
    # Zoom / Gesamtansicht
    # ========================================================


    # Query + alle Top-K Punkte berücksichtigen
    all_lons = np.concatenate([
        [query["lon"]],
        top_k["lon"].to_numpy()
    ])

    all_lats = np.concatenate([
        [query["lat"]],
        top_k["lat"].to_numpy()
    ])

    min_lon = all_lons.min()
    max_lon = all_lons.max()
    min_lat = all_lats.min()
    max_lat = all_lats.max()

    # Etwas Rand um die Punkte
    lon_range = max_lon - min_lon
    lat_range = max_lat - min_lat

    # Falls die Punkte sehr nah beieinander liegen
    lon_range = max(lon_range, 0.005)
    lat_range = max(lat_range, 0.005)

    padding = 0.25

    lon_center = (min_lon + max_lon) / 2
    lat_center = (min_lat + max_lat) / 2

    half_lon = lon_range / 2 * (1 + padding)
    half_lat = lat_range / 2 * (1 + padding)

    ax.set_xlim(
        lon_center - half_lon,
        lon_center + half_lon
    )

    ax.set_ylim(
        lat_center - half_lat,
        lat_center + half_lat
    )

    ax.set_aspect(
        1 / np.cos(np.radians(lat_center))
    )

    ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()


# ============================================================
# 6. DETAILANSICHT
# ============================================================

plot_retrieval_map(
    street_net=street_net,
    city_polygon=city_polygon,
    query=query,
    top_k=top_k,
    zoom=True,
    title=f"Query + Top-{K} Retrieval — Detailansicht",
)


# ============================================================
# 7. GANZ OSNABRÜCK
# ============================================================

plot_retrieval_map(
    street_net=street_net,
    city_polygon=city_polygon,
    query=query,
    top_k=top_k,
    zoom=False,
    title=f"Query + Top-{K} Retrieval — Osnabrück",
)
